In [0]:
# ==========================================
# 01_Raw_to_Bronze.py
# Member 1: Ingestion & Bronze Layer
# ==========================================

from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, IntegerType, ArrayType
)
from pyspark.sql.functions import current_timestamp, lit

# 1. Define Strict Schema (Schema-on-Read)
# This perfectly matches the 5-city JSON payload.

# Define the structure for the "hourly_units" object
hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("carbon_monoxide", StringType(), True),
    StructField("carbon_dioxide", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nitrogen_dioxide", StringType(), True)
])

# Define the structure for the "hourly" object (arrays of data)
hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("carbon_monoxide", ArrayType(DoubleType()), True),
    StructField("carbon_dioxide", ArrayType(DoubleType()), True), # Casted to Double for safety, even if API sends Ints
    StructField("pm2_5", ArrayType(DoubleType()), True),
    StructField("pm10", ArrayType(DoubleType()), True),
    StructField("nitrogen_dioxide", ArrayType(DoubleType()), True)
])

# Define the main root structure of the JSON
open_meteo_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
    StructField("city", StringType(), True),
    StructField("source_system", StringType(), True)
])

print("Strict Schema defined successfully and matches the verified JSON payload!")

In [0]:
# ==========================================
# Sanity Check: Test the Schema against the file
# ==========================================

# Replace this string with the actual path to your JSON file in Databricks
file_path = "/Volumes/workspace/default/bronze/incremental_load_sample.json"

try:
    # Read the JSON file using our strict schema and the multiline option 
    # (multiline=True is required because the JSON array spans multiple lines)
    test_df = (spark.read
               .schema(open_meteo_schema)
               .option("multiline", "true")
               .json(file_path))
    
    print("File read successfully! Here is a preview:")
    display(test_df)
    
except Exception as e:
    print("Error reading the file. Check your file path!")
    print(e)

In [0]:
# ==========================================
# Create Audit Log Table (Run once)
# ==========================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_table_path = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/YOUR_VOLUME/pipeline_execution_logs"

# Define the schema based exactly on the Phase 2 requirements
log_schema = StructType([
    StructField("layer_processed", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", IntegerType(), True)
])

# Create an empty DataFrame and save it as a Delta table (if it doesn't exist)
try:
    spark.read.format("delta").load(log_table_path)
    print("Log table already exists.")
except:
    empty_df = spark.createDataFrame([], log_schema)
    empty_df.write.format("delta").save(log_table_path)
    print("Audit Log table created successfully!")

# ==========================================
# FINAL Parameterized Bronze Ingestion Function (With Logging)
# ==========================================
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
from pyspark.sql import functions as F

def ingest_raw_to_bronze(input_file_path, bronze_table_path, log_table_path):
    layer_name = "Raw-to-Bronze"
    start_time = datetime.now()
    status = "Failed"
    row_count = 0
    
    print(f"--- Starting {layer_name} ---")
    
    try:
        # 1. Read & Add Metadata
        raw_df = (spark.read
                  .schema(open_meteo_schema)
                  .option("multiline", "true")
                  .json(input_file_path))
        
        bronze_df = (raw_df
                     .withColumn("load_timestamp", current_timestamp())
                     .withColumn("source_file", F.col("_metadata.file_path")))
        
        # 2. Write to Bronze
        (bronze_df.write
         .format("delta")
         .mode("append") 
         .option("mergeSchema", "true") 
         .save(bronze_table_path))
        
        row_count = bronze_df.count()
        status = "Success"
        print(f"Success! {row_count} rows appended to Bronze Layer.")
        
    except Exception as e:
        print(f"Pipeline Failed: {e}")
        status = f"Failed: {str(e)[:50]}" # Log first 50 chars of error
        
    finally:
        end_time = datetime.now()
        
        # 3. Write to Audit Log (Fulfills Phase 2 Audit Requirement)
        log_entry = spark.createDataFrame([(
            layer_name, 
            input_file_path, 
            start_time, 
            end_time, 
            status, 
            row_count
        )], schema=log_schema)
        
        log_entry.write.format("delta").mode("append").save(log_table_path)
        print("Audit Log updated.")
        print(f"---------------------------------")
        
    return status

In [0]:
# ==========================================
# 3. Execute the Pipeline (Simulating a run)
# ==========================================

# 1. Define your parameters
# (Change these paths to your actual Unity Catalog Volume paths)
source_data_parameter = "/Volumes/workspace/default/bronze/incremental_load_sample.json"
bronze_delta_path = "/Volumes/workspace/default/bronze_layer/bronze_carbon_data"
log_table_path = "/Volumes/workspace/default/bronze_layer/pipeline_execution_logs"

ingest_raw_to_bronze(source_data_parameter, bronze_delta_path, log_table_path)

# Verify the log was written!
display(spark.read.format("delta").load(log_table_path))